## Passage Retrieval


In [ ]:
# %pip install openai

In [18]:
# %pip install --upgrade langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 727.6/727.6 kB 25.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.4/381.4 kB 29.3 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 0.0.136
    Uninstalling langchain-0.0.136:
      Successfully uninstalled langchain-0.0.136
Note: you may need to restart the kernel to use updated packages.


In [75]:
# %pip install chromadb

In [77]:
# %pip install tiktoken

In [ ]:
import pandas as pd
import os

### Passage Retrieval
- Use Case: Given a question (from various question categories), return at least one relevant answer, where each answer is tied to at least one relevant passage

- Datasets: 50q-benchmark-03282023.csv; respiratory

- Evaluation set: 50 questions, together with answers and relevant passage (generated by MDs using the Calibration app)

- KPIs: 
- - Recall / Precision at 1 > 0.50
- - Recall / Precision at 3 > 0.85

### High-level steps

- Parse csv files to generate topic text files, and, assuming passage = section, topic-section text files.
- Separately create indexes for topics and topic-sections using OpenAI embeddings, store embeddings in vector database (many alternatives exist here, including using a traditional search index for the retriever).
- Ask a question of either index, returning top k most relevant answers, with the topic-section associated with each

#### Generate topic and topic-section text files
For each topic csv file,
- grab topic id
- grab topic title
- merge text across all rows within topic to generate topic text, within section_id to generate topic-section text
- write distinct topic topic-section text files with topic id, topic title, and merged orig_text

In [43]:
# make destination folder
os.mkdir('data/topics-txt')
os.mkdir('data/topic-sections-txt')

In [46]:
# review sample file
csv = '11.csv'
df = pd.read_csv(f'data/topics-csv-11142022/{csv}')

In [47]:
df.head()

,topic_id,topic_version,subj_topic_id,section_id,section_hierarchy,para_id,text,orig_text,sflf_text,graphic_label_title,graphic_label_refid
0,11,16.0,GAST/11,NaN,NaN,NaN,Helicobacter pylori (h pylori) and gastroesop...,Helicobacter pylori and gastroesophageal reflu...,NaN,NaN,NaN
1,11,16.0,GAST/11,H1,INTRODUCTION,NaN,INTRODUCTION,INTRODUCTION,NaN,NaN,NaN
2,11,16.0,GAST/11,H1,NaN,0.0,Helicobacter pylori (h pylori) (H. pylori) i...,Helicobacter pylori (H. pylori) is an importan...,GERD: Gastroesophageal reflux disease,NaN,NaN
3,11,16.0,GAST/11,H1,NaN,1.0,This topic review will summarize the availabl...,This topic review will summarize the available...,NaN,NaN,NaN
4,11,16.0,GAST/11,H2,EPIDEMIOLOGY,NaN,EPIDEMIOLOGY,EPIDEMIOLOGY,NaN,NaN,NaN


In [28]:
# section_ids = sorted(set(list(df['section_id'].dropna().values)))
# section_ids

In [51]:
# sid = 'H1'
# # section_text = '\n'.join(df.loc[df['section_id'] == sid, 'text'].values)
# section_text = '\n'.join(df.loc[df['section_id'] == sid, 'text'].dropna())
# section_text

In [30]:
# topic_id = str(df['topic_id'].values[0])
# topic_id

In [31]:
# topic_title = df['text'].values[0]
# topic_title

In [27]:
# fpath = f'data/topic-sections/{topic_id}-{sid}.txt'
# with open(fpath, 'a') as f:
#     print(f'topicId: {topic_id}', file=f)
#     print(f'topicTitle: {topic_title}', file=f)
#     print(f'\n{section_text}', file=f)

In [53]:
# topic_text = '\n'.join(df['text'].dropna().values)
# topic_text

##### Generate Topic text files

In [55]:
# generate topic text files
for csv in os.listdir('data/topics-csv-11142022'):
    if csv[-3:] != 'csv': continue
    # tid = csv.split('.')[0]
    # if tid < '5624': continue
    # load csv into df
    df = pd.read_csv(f'data/topics-csv-11142022/{csv}')
    # grab topic id
    topic_id = str(df['topic_id'].values[0])
    print(topic_id)
    # grab topic title
    topic_title = df['text'].values[0]

    # join topic text, write file
    topic_text = '\n'.join(df['text'].dropna().values)

    fpath = f'data/topics-txt/{topic_id}.txt'
    with open(fpath, 'a') as f:
        print(f'topicId: {topic_id}', file=f)
        print(f'topicTitle: {topic_title}', file=f)
        print(f'\n{topic_text}', file=f) 

100191
101088
101773
104135
105590
106357
106496
11
110530
112420
1130
113082
113084
113103
116
116196
117222
118590
119911
120681
120959
121444
121739
123
126585
127099
127985
128470
128931
130052
1303
133139
13516
13536
13590
1361
1376
1379
1380
1388
138854
13893
13917
13927
14113
14202
14219
14255
1428
14428
14611
1468
14688
14706
14894
14953
14965
14972
15005
15006
15008
15011
15017
1506
15075
15165
15176
15261
1529
15325
15400
15592
15687
15695
15781
15848
15860
15892
15918
16109
16199
16377
16503
1690
16924
17006
17008
17122
1733
1766
1774
1779
1800
1809
1897
2011
2071
2073
2078
2087
2152
2212
2214
2219
2220
2227
2228
2231
2233
2234
2239
2241
2244
2257
2258
2263
2265
2269
2274
2278
2398
2415
2472
2473
2478
2494
2502
2511
2526
2530
2537
2569
2587
2601
2633
2655
2663
2665
2671
2713
2770
2842
287
296
3249
3263
3288
3311
3316
3333
3337
3343
3344
3346
3347
3349
3354
3355
3357
3360
3363
3497
3558
3753
3875
393
4065
4069
411
433
438
441
4452
4561
473
4794
4801
4807
4893
4895
4896
4899
4

In [56]:
# how many topic files
len(os.listdir('data/topics-txt'))

393

##### Generate Topic-Section text files

In [208]:
# generate topic-sections
for csv in os.listdir('data/topics-csv-11142022'):
    if csv[-3:] != 'csv': continue
    # tid = csv.split('.')[0]
    # if tid < '5624': continue
    # load csv into df
    df = pd.read_csv(f'data/topics-csv-11142022/{csv}')
    # grab topic id
    topic_id = str(df['topic_id'].values[0])
    print(topic_id)
    # grab topic title
    topic_title = df['text'].values[0]
    # get the unique section_ids
    section_ids = sorted(set(list(df['section_id'].dropna().values)))
    
    # for each topic-section pair, join section text, write file
    for sid in section_ids:
        section_title = df.loc[df['section_id'] == sid, 'text'].dropna().values[0]
        section_text = '\n'.join(df.loc[df['section_id'] == sid, 'text'].dropna().values[1:])
        fpath = f'data/topic-sections-txt/{topic_id}-{sid}.txt'
        with open(fpath, 'a') as f:
            print(f'topicId: {topic_id}', file=f)
            print(f'topicTitle: {topic_title}', file=f)
            print(f'sectionId: {sid}', file=f)
            print(f'sectionTitle: {section_title}', file=f)
            print(f'\n{section_text}', file=f)         

100191
101088
101773
104135
105590
106357
106496
11
110530
112420
1130
113082
113084
113103
116
116196
117222
118590
119911
120681
120959
121444
121739
123
126585
127099
127985
128470
128931
130052
1303
133139
13516
13536
13590
1361
1376
1379
1380
1388
138854
13893
13917
13927
14113
14202
14219
14255
1428
14428
14611
1468
14688
14706
14894
14953
14965
14972
15005
15006
15008
15011
15017
1506
15075
15165
15176
15261
1529
15325
15400
15592
15687
15695
15781
15848
15860
15892
15918
16109
16199
16377
16503
1690
16924
17006
17008
17122
1733
1766
1774
1779
1800
1809
1897
2011
2071
2073
2078
2087
2152
2212
2214
2219
2220
2227
2228
2231
2233
2234
2239
2241
2244
2257
2258
2263
2265
2269
2274
2278
2398
2415
2472
2473
2478
2494
2502
2511
2526
2530
2537
2569
2587
2601
2633
2655
2663
2665
2671
2713
2770
2842
287
296
3249
3263
3288
3311
3316
3333
3337
3343
3344
3346
3347
3349
3354
3355
3357
3360
3363
3497
3558
3753
3875
393
4065
4069
411
433
438
441
4452
4561
473
4794
4801
4807
4893
4895
4896
4899
4

In [209]:
# how many topic-section files
len(os.listdir('data/topic-sections-txt'))

11672

#### Create Index on topic files

In [1]:
from langchain.document_loaders import DirectoryLoader
from langchain.document_loaders import TextLoader

In [2]:
topics_loader = DirectoryLoader('data/topics-txt', glob="*.txt", loader_cls=TextLoader)

In [3]:
topic_docs = topics_loader.load()

In [4]:
from langchain.indexes import VectorstoreIndexCreator

##### Set environment variables such as OPENAI_API_KEY
Typically not done in plain text in notebook...

In [5]:
import os, openai
os.environ["OPENAI_API_TYPE"] = openai.api_type = "azure"
os.environ["OPENAI_API_VERSION"] = openai.api_version = "2022-12-01"
# os.environ["OPENAI_API_BASE"] = openai.api_base = "https://[your_service_name].openai.azure.com/"
os.environ["OPENAI_API_BASE"] = openai.api_base = "https://openai-skyblue-east.openai.azure.com/"
# os.environ["OPENAI_API_KEY"] = openai.api_key = "[your_api_key]" 
os.environ["OPENAI_API_KEY"] = openai.api_key = "d1217bdda09b41bc906780b7019d5077" 
# https://openai-skyblue-east.openai.azure.com/

In [6]:
# define the model that will be used to generate document embeddings to store in vector db
from chromadb.utils import embedding_functions
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
                api_key=os.environ["OPENAI_API_KEY"],
                model_name="text-embedding-ada-002"
            )

In [7]:
#split every topic document into chunks of size that the qa model can handle, ~4000
# from langchain.text_splitter import CharacterTextSplitter
from langchain.text_splitter import TokenTextSplitter
text_splitter = TokenTextSplitter(chunk_size=3200, chunk_overlap=200)
topic_texts = text_splitter.split_documents(topic_docs)

In [8]:
len(topic_texts)

1492

In [108]:
# topic_texts[0]

In [81]:
# topics_index = VectorstoreIndexCreator().from_loaders([topics_loader])

In [9]:
#generate embeddings for each chunk
import time, tqdm
topic_embeddings = []

for text in tqdm.tqdm(topic_texts):
    # split into chunks if necessary
    
    try:
        response = openai.Embedding.create(
            input=text.page_content,
            engine="text-embedding-ada-002")
        emb = response['data'][0]['embedding']
        topic_embeddings.append(emb)
    except Exception as e:
        time.sleep(8)
        response = openai.Embedding.create(
            input=text.page_content,
            engine="text-embedding-ada-002")
        emb = response['data'][0]['embedding']
        topic_embeddings.append(emb)

  0%|          | 1/1492 [00:08<3:23:51,  8.20s/it]


RateLimitError: Requests to the Embeddings_Create Operation under Azure OpenAI API version 2022-12-01 have exceeded call rate limit of your current OpenAI S0 pricing tier. Please retry after 52 seconds. Please go here: https://aka.ms/oai/quotaincrease if you would like to further increase the default rate limit.

In [10]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma

# initialize the vector store and link an OpenAI embedding model
azure_openai_embeddings = OpenAIEmbeddings(
    document_model_name="text-embedding-ada-002", query_model_name="text-embedding-ada-002")
topics_vectorstore = Chroma("topics_collection", embedding_function=azure_openai_embeddings)

Using embedded DuckDB without persistence: data will be transient


In [169]:
topics_search = Chroma.from_texts(
    texts=[topic_texts[i].page_content for i in range(len(topic_texts))], 
    embedding_function=azure_openai_embeddings,
    embeddings=topic_embeddings, 
    metadatas=[text.metadata for text in topic_texts]
    ).as_retriever(search_kwargs={"k": 10})

Using embedded DuckDB without persistence: data will be transient
No embedding_function provided, using default embedding function: SentenceTransformerEmbeddingFunction


In [170]:
relevant_docs = topics_search.get_relevant_documents(q)

In [214]:
# add the topic documents to the vector store
topics_vectorstore._collection.add(
    ids= [f"doc_{i}" for i in range(len(topic_texts))],
    documents=[topic_texts[i].page_content for i in range(len(topic_texts))],
    embeddings=topic_embeddings,
    metadatas=[text.metadata for text in topic_texts]
)

In [215]:
rel_docs = topics_vectorstore.as_retriever(search_kwargs={"k": 10}).get_relevant_documents(q)

In [216]:
len(rel_docs)

10

In [221]:
for doc in rel_docs:
    print(doc.metadata['source'])

data/topics-txt/4896.txt
data/topics-txt/97581.txt
data/topics-txt/4896.txt
data/topics-txt/4893.txt
data/topics-txt/4893.txt
data/topics-txt/97581.txt
data/topics-txt/4899.txt
data/topics-txt/4896.txt
data/topics-txt/4900.txt
data/topics-txt/4900.txt


In [19]:
from langchain.chains import RetrievalQAWithSourcesChain, RetrievalQA
from langchain.llms import AzureOpenAI
from langchain.chat_models import AzureChatOpenAI

# llm = AzureChatOpenAI(
#     model_name="gpt-35-turbo", 
#     # engine="gpt-35-turbo", 
#     temperature=0.0)
llm = AzureChatOpenAI(
    deployment_name="gpt-35-turbo",
    # model_name="gpt-35-turbo", 
    # model_name="gpt-3.5-turbo", # model_name may have to match OAI model_names
    temperature=0,
    openai_api_version="2023-03-15-preview",
)

# llm=AzureOpenAI(
#     deployment_name="text-davinci-003", 
#     model_name="text-davinci-003")    

# rate-limiting will make creating the embedding vectors take 20+ hours, so won't do this
# qa = RetrievalQAWithSourcesChain.from_chain_type(
#     llm=llm, 
#     chain_type="map_reduce", 
#     # chain_type="stuff",
#     retriever=topics_vectorstore.as_retriever(search_kwargs={"k":10}), 
#     # retriever=topics_search,
#     # reduce_k_below_max_tokens=True,
#     return_source_documents=True,
#     )

In [12]:
q = 'What is preferred initial pharmacotherapy for Parkinson Disease?'
q = 'What is the preferred initial treatment for bacterial vaginosis?'
q = 'How is BPH diagnosed?'

In [20]:
from langchain.schema import HumanMessage
print(llm([HumanMessage(content=q)]))

KeyError: 'content'

In [188]:
rel_docs = topics_vectorstore.as_retriever().get_relevant_documents(q)

In [240]:
result = qa({"question": q}, return_only_outputs=True)

In [241]:
print(result['answer'])
[r.metadata['source'] for r in result['source_documents']]

 BPH is typically diagnosed by a medical history, physical examination, and laboratory tests such as a urinalysis and prostate-specific antigen (PSA). Urodynamic testing may be used in certain cases to help guide therapy.
Source: data/topics-txt/6889.txt, data/topics-txt/8093.txt, data/topics-txt/6890.txt, data/topics-txt/6879.txt, data/topics-txt/8085.txt


['data/topics-txt/6889.txt',
 'data/topics-txt/8093.txt',
 'data/topics-txt/8093.txt',
 'data/topics-txt/6890.txt',
 'data/topics-txt/6879.txt',
 'data/topics-txt/6891.txt',
 'data/topics-txt/8093.txt',
 'data/topics-txt/6889.txt',
 'data/topics-txt/8085.txt',
 'data/topics-txt/8085.txt']

In [235]:
# load all questions and collect results
q50df = pd.read_csv('data/50q-benchmark-03282023.csv')

In [238]:
q50 = list(set(q50df['query'].values))
q50

['What is the treatment for anovulation due to PCOS?',
 'What are the common side effects of obesity pharmacotherapy?',
 'What is the preferred pharmacotherapy for obesity?',
 'Does bacterial vaginosis always require antibiotic treatment?',
 'What foods commonly trigger GERD symptoms?',
 'Can metformin induce weight loss in PCOS?',
 'What are the common causes of headache in children presenting to the Emergency department',
 'What are common causes of temporal lobe epilepsy?',
 'how effective is bariatric surgery for management of obesity',
 'What is the preferred initial treatment for bacterial vaginosis?',
 'When is surgery indicated for BPH?',
 'what are the signs and symptoms of PCOS',
 'does bacterial vaginosis have adverse effects on pregnancy outcomes',
 'In which patients is MRI appropriate for breast cancer screening?',
 'What are common symptoms of PCOS in adolesents?',
 'At what age should breast cancer screening begin?',
 'Is dietary modification recommended for management 

#### Create Index on topic section files